In [1]:
import os
import json
import random
from typing import List, Dict, Any

In [2]:
def format_nl_premises(premises_nl: List[str]) -> str:
    """Format premises as a numbered list string with 0-based bracketed indices."""
    return "\n".join(f"[{i}] {premise}" for i, premise in enumerate(premises_nl))

def format_fol_premises(premises_fol: List[str]) -> List[str]:
    """Format FOL premises."""
    return [premise.strip() for premise in premises_fol if premise.strip()]

def build_unified_samples(record: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Build conversational message formats for a single-pass joint-output model.
    This formats the output as a <think> block followed by a JSON object,
    supporting correct 0-based indexing and logical fallacies prevention.
    """

    premises = record.get("premises", [])
    premises_fol = record.get("premises-FOL", [])
    
    questions = record.get("question", [])
    options_list = record.get("options", [])
    answers = record.get("answer", [])
    explanations = record.get("explanation", [])
    idx_list = record.get("idx", [])
    
    # System Prompt aligned with EXACT 2026 rules and JSON output schema
    system_prompt = (
        "You are a rigorous logical reasoning AI expert specializing in First-Order Logic (FOL) for EXACT 2026.\n"
        "Your task is to analyze the given natural language premises, construct formal FOL representations, deduce the logical conclusion, and identify the exact premises used.\n\n"
        "Output format: You MUST return a single, valid JSON object with keys: \"answer\", \"unit\", \"explanation\", \"premises_used\", \"reasoning\".\n\n"
        "Strict Rules:\n"
        "1. \"answer\" format:\n"
        "   - If \"options\" is non-empty, the value of \"answer\" MUST EXACTLY match one of the listed options.\n"
        "   - If Options is empty ([]), the answer is free-form (a number or a short text).\n"
        "2. \"unit\" format:\n"
        "   - Must always be an empty string \"\" for logic queries.\n"
        "3. \"premises_used\" tracking:\n"
        "   - In the input query, each premise is prefixed with its 0-based index in brackets, like \"[i] Premise content\".\n"
        "   - You MUST extract the exact 0-based indices from these brackets for the premises that are strictly necessary for the deduction, and return them as a list of integers in \"premises_used\".\n"
        "   - If the answer is \"Uncertain\" or \"Unknown\" due to a missing condition, only list the premises that lead to the point of uncertainty.\n"
        "4. \"explanation\" format:\n"
        "   - A concise, natural language explanation detailing step-by-step how the premises lead to the final answer.\n"
        "5. \"reasoning\" format:\n"
        "   - An object of shape {\"type\": \"fol\", \"steps\": [...]}.\n"
        "   - In \"steps\", write down the formal FOL translations (using ForAll, Exists, Implies, And, Or, Not syntax) and the intermediate derivation steps.\n"
        "6. Avoid Logical Fallacies:\n"
        "   - Denying the Antecedent: If a rule says \"If A then B\", and you know \"Not A\", you CANNOT conclude \"Not B\". The status is Uncertain/Unknown.\n"
        "   - Affirming the Consequent: If a rule says \"If A then B\", and you know \"B\", you CANNOT conclude \"A\"."
    )
    
    formatted_premises = format_nl_premises(premises)
    formatted_premises_fol = format_fol_premises(premises_fol)
    
    samples = []
    
    for i in range(len(questions)):
        query = questions[i]
        options = options_list[i]
        answer = answers[i]
        explanation = explanations[i]
        
        # Convert 1-based indices from SFT dataset to 0-based indices
        raw_used = idx_list[i]
        premises_used = [idx - 1 for idx in raw_used] if any(idx > 0 for idx in raw_used) else raw_used
        
        # Let's map Z3/FOL steps into FOL reasoning
        fol_steps = list(formatted_premises_fol)
        
        # User Prompt
        user_content = (
            f"Premises:\n{formatted_premises}\n\n"
            f"Question: {query}\n"
            f"Options: {json.dumps(options, ensure_ascii=False)}"
        )
        
        # Build the JSON response object
        json_output = {
            "answer": answer.strip(),
            "unit": "",
            "explanation": explanation.strip(),
            "premises_used": premises_used,
            "reasoning": {
                "type": "fol",
                "steps": fol_steps
            }
        }
        
        # Format Assistant content: <think> reasoning first, then the JSON block
        think_content = f"Explanation of reasoning: {explanation.strip()}\nSelected premises: {json.dumps(premises_used)}"
        json_str = json.dumps(json_output, indent=2, ensure_ascii=False)
        
        assistant_content = (
            f"<think>\n{think_content}\n</think>\n"
            f"{json_str}"
        )
        
        samples.append({
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": assistant_content}
            ]
        })
        
    return samples

def prepare_data(
    input_file: str,
    output_dir: str,
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    seed: int = 42
):
    """Load, process, shuffle and split dataset at the sample level (after flattening)."""
    print(f"Loading data from {input_file}...")
    with open(input_file, "r", encoding="utf-8") as f:
        records = json.load(f)
    
    samples = []
    for rec in records:
        samples.extend(build_unified_samples(rec))
        
    print(f"Processed and flattened into {len(samples)} single-query samples successfully.")
    
    # Random shuffle and split
    random.seed(seed)
    indices = list(range(len(samples)))
    random.shuffle(indices)
    
    total = len(samples)
    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)
    
    train_idx = indices[:train_end]
    val_idx = indices[train_end:val_end]
    test_idx = indices[val_end:]
    
    print(f"Split: Train={len(train_idx)}, Val={len(val_idx)}, Test={len(test_idx)}")
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    splits = {
        "logic_train.jsonl": train_idx,
        "logic_val.jsonl": val_idx,
        "logic_test.jsonl": test_idx
    }
    for filename, idx_list in splits.items():
        filepath = os.path.join(output_dir, filename)
        with open(filepath, "w", encoding="utf-8") as f:
            for idx in idx_list:
                f.write(json.dumps(samples[idx], ensure_ascii=False) + "\n")
        print(f"Saved {len(idx_list)} samples to {filepath}")

In [3]:
workspace_dir = r"d:\Education\exact_2026"
input_json = os.path.join(workspace_dir, "data", "processed", "Logic_SFT_with_options.json")
output_directory = os.path.join(workspace_dir, "data", "processed")

prepare_data(
    input_file=input_json,
    output_dir=output_directory,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    seed=42
)